# 03 — Cellpose-SAM parameter tuning (macrophage ∥ candida foreground)
Typed-param tuning (no ipywidgets — those don't render under OOD JupyterHub). Run top-to-bottom on a **gpuq A30/A100** node with the **`canmac (pixi)`** kernel.

## 0 — Environment & GPU sanity (fail loud before any inference)

In [ ]:
import os

# Proves Plan-00's kernel env block is in effect (else cellpose would try an offline download).
print("CELLPOSE_LOCAL_MODELS_PATH =", os.environ.get("CELLPOSE_LOCAL_MODELS_PATH"))

import torch
assert torch.cuda.is_available(), "No CUDA — request a gpuq A30/A100 node under OOD (D-03)"
dev = torch.cuda.get_device_name(0)
print("CUDA device =", dev)
# Pitfall 1: gpuq is heterogeneous (P100 + A30 + A100); the 12 GB P100 OOMs on cpsam.
assert 'P100' not in dev, f"On {dev!r}: the P100 OOMs on cpsam — request an A30/A100"

In [ ]:
# --- repo-root bootstrap: OOD kernel CWD is notebooks/, not the repo root ---
# canmac isn't pip-installed; make the repo root importable AND the CWD (so the
# reader's relative results/ + params/ paths resolve like the batch stage does).
import sys, os, pathlib
_here = pathlib.Path.cwd()
for _root in (_here, *_here.parents):
    if (_root / 'canmac' / '__init__.py').exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
else:
    _root = pathlib.Path('/vast/scratch/users/kriel.j/monash_lsm'); sys.path.insert(0, str(_root))
os.chdir(_root)

 # pick up canmac edits on re-run (no kernel restart)
import json
import numpy as np
import matplotlib.pyplot as plt

# The SAME reader + inference path the batch stage uses (Pattern 3 — no config drift).
from canmac.io.reader import get_view
from canmac.stages.segment import segment_timepoint
print("cwd =", pathlib.Path.cwd())

## 1 — Tuning helpers (typed params, static overlay — no widgets)

In [ ]:
# Static (no-widget) QC overlay — renders under OOD JupyterHub (interactive sliders do not).
from canmac.stages.segment import preprocess_volume

def show_overlay(raw, pre, labels, zs=None, title=""):
    """raw MIP | de-haloed MIP | de-haloed MIP + label boundaries, plus a mid Z-slice."""
    zmax = raw.shape[0]
    zs = zs or [zmax // 2]
    ncol = 3 + len(zs)
    fig, axes = plt.subplots(1, ncol, figsize=(4 * ncol, 4))
    axes[0].imshow(raw.max(0), cmap="gray"); axes[0].set_title(f"{title}\nraw MIP"); axes[0].axis("off")
    axes[1].imshow(pre.max(0), cmap="gray"); axes[1].set_title("pre-processed MIP"); axes[1].axis("off")
    axes[2].imshow(pre.max(0), cmap="gray"); axes[2].set_title("pre + mask")
    if labels is not None:
        axes[2].contour(labels.max(0) > 0, colors="yellow", linewidths=0.5)
    axes[2].axis("off")
    for ax, z in zip(axes[3:], zs):
        ax.imshow(pre[z], cmap="gray"); ax.set_title(f"z={z}")
        if labels is not None:
            ax.contour(labels[z] > 0, colors="yellow", linewidths=0.5)
        ax.axis("off")
    plt.tight_layout(); plt.show()


def tune(dataset, channel, t, params, save=False):
    """Segment ONE timepoint with `params`; show raw-vs-de-haloed-vs-mask; optionally save.

    params['preprocess'] (none|tophat|dog) suppresses the candida halo BEFORE Cellpose —
    the SAME code path the batch uses (canmac.stages.segment.preprocess_volume). Edit the
    PARAMS dict, re-run this cell; the model is cached across calls. save=True writes
    params -> params/cpsam_{channel}.json (the sidecar the batch reads).
    """
    correct = channel == "macrophage"  # D-05: macrophage corrected, candida raw
    raw = np.ascontiguousarray(get_view(dataset, channel, t, correct=correct).compute()).astype(np.float32)
    pre = preprocess_volume(raw, params.get("preprocess", "none"),
                            tophat_radius=params.get("tophat_radius", 10),
                            dog_low=params.get("dog_low", 1.0), dog_high=params.get("dog_high", 6.0))
    labels = segment_timepoint(dataset, channel, t, params)  # applies the SAME preprocess internally
    n = int(labels.max())
    print(f"{dataset}/{channel} t{t:03d}: n_objects={n}  params={params}")
    show_overlay(raw, pre, labels, title=f"{dataset}/{channel} t{t} (n={n})")
    if save:
        pth = f"params/cpsam_{channel}.json"; json.dump(params, open(pth, "w"), indent=2); print("saved ->", pth)
    return labels


# --- shared per-object diagnostics (used by BOTH the threshold prototype and micro-sam) ---
def object_features(labels, vox=0.145, len_um_hyphae=3.0):
    """Per-object skeleton features -> (feats dict {label:(len_um,branched,is_hyphae)}, DataFrame).
    hyphae = branched OR skeleton length >= len_um_hyphae. Same rule as the threshold prototype."""
    import pandas as pd
    from skimage.measure import regionprops
    from skimage.morphology import skeletonize
    from skan import Skeleton, summarize
    feats, rows = {}, []
    for pr in regionprops(labels):
        z0, y0, x0, z1, y1, x1 = pr.bbox
        sub = (labels[z0:z1, y0:y1, x0:x1] == pr.label)
        sk = skeletonize(sub)
        L, junc = 0.0, False
        if sk.sum() > 1:
            try:
                df = summarize(Skeleton(sk, spacing=(vox, vox, vox)), separator="_")
                L = float(df["branch_distance"].sum()); junc = bool((df["branch_type"] >= 1).any() and len(df) > 1)
            except Exception:
                L = float(sk.sum()) * vox
        ishy = junc or (L >= len_um_hyphae)
        feats[pr.label] = (round(L, 2), junc, ishy)
        rows.append({"obj": pr.label, "vox": int(pr.area), "skel_len_um": round(L, 2),
                     "branched": junc, "PROPOSED_class": "hyphae" if ishy else "yeast"})
    cols = ["obj","vox","skel_len_um","branched","PROPOSED_class"]
    df = pd.DataFrame(rows, columns=cols)          # empty-safe: keeps columns when 0 objects
    if df.empty:
        print("WARNING: 0 objects — knobs too strict (lower foreground_threshold / min_z_extent / min_size).")
        return feats, df
    return feats, df.sort_values("vox", ascending=False)

def show_object_panels(raw, labels, feats, title="", max_panels=16):
    """Per-object panels: raw crop (MIP) + cyan mask contour, titled with the PROPOSED class."""
    from skimage.measure import regionprops
    props = sorted(regionprops(labels), key=lambda p: p.area, reverse=True)[:max_panels]
    if not props:
        print("no objects to panel"); return
    nc = 4; nr = int(np.ceil(len(props) / nc))
    fig, axes = plt.subplots(nr, nc, figsize=(4 * nc, 4 * nr)); axes = np.atleast_1d(axes).ravel()
    for a, pr in zip(axes, props):
        z0, y0, x0, z1, y1, x1 = pr.bbox
        a.imshow(raw[z0:z1, y0:y1, x0:x1].max(0), cmap="gray")
        a.contour((labels[z0:z1, y0:y1, x0:x1] == pr.label).max(0), colors="cyan", linewidths=0.6)
        L, junc, ishy = feats.get(pr.label, (0, False, False))
        a.set_title(f"obj {pr.label}: {int(pr.area)}vox {L}um br={junc}\n-> PROPOSED {'hyphae' if ishy else 'yeast'}", fontsize=9)
        a.axis("off")
    for a in axes[len(props):]: a.axis("off")
    plt.suptitle(title, y=1.002); plt.tight_layout(); plt.show()

## 2 — Macrophage (ch3) — defaults already look good; adjust if needed

In [ ]:
# Loads your current saved params so re-running never clobbers earlier tuning. Edit + re-run.
MACRO_PARAMS = json.load(open("params/cpsam_macrophage.json"))
# e.g. MACRO_PARAMS["cellprob_threshold"] = 0.0
print(MACRO_PARAMS)

In [ ]:
labels_macro = tune("ROI2", "macrophage", 90, MACRO_PARAMS, save=True)

## 3 — Candida foreground (ch2) — **needs tuning** (foreground only)

In [ ]:
# ==== Candida FOREGROUND — halo suppression (Phase-4 does yeast/hyphae split, not here). ====
# The fluorescence has a diffuse PSF 'halo' -> Cellpose over-grows blobby masks. Knobs:
#   preprocess : "tophat" (3D white top-hat; removes broad halo; tophat_radius ~ a bit
#                larger than a yeast cell in voxels, ~10-15) | "dog" (band-pass; FAST) | "none"
#   cellprob_threshold : RAISE (+1..+3) to tighten masks to bright cores
#   flow3D_smooth      : raise (1-2) if one object fragments across Z
# Starts from your saved candida params; try preprocess="tophat" first, then compare "dog".
CANDIDA_PARAMS = json.load(open("params/cpsam_candida.json"))
CANDIDA_PARAMS["preprocess"] = "tophat"   # <-- the halo experiment; switch to "dog" / "none" to compare
CANDIDA_PARAMS.setdefault("tophat_radius", 10)
print(CANDIDA_PARAMS)

In [ ]:
labels_cand = tune("ROI2", "candida", 90, CANDIDA_PARAMS, save=True)

## 4 — Optional napari 3D spot-check (or use view_labels.py on VNC)

## 3b — Candida via threshold → skeleton graph (PROTOTYPE — no Cellpose, CPU-only)
Threshold the foreground → **skeletonize** → **skan** graph (branch length + junctions) to propose filamentous (hyphae) vs compact (yeast). **Everything below is PROPOSED — confirm each object in the panels + table before trusting it; counts alone do not validate the split.** Tune `preprocess`, `min_size`, `len_um_hyphae`; use the napari cell for 3D.

In [ ]:
# PROTOTYPE (throwaway, CPU-only): candida foreground -> 3D skeleton -> skan graph -> yeast|hyphae.
# ALL results are PROPOSED — confirm each object in the panels/table below. Counts != validation.
import pandas as pd, matplotlib.colors as mcolors
from IPython.display import display
from skimage.filters import threshold_otsu
from skimage.morphology import skeletonize, remove_small_objects
from skimage.measure import label as cc_label, regionprops
from skan import Skeleton, summarize

def candida_split(t=90, preprocess="dog", tophat_radius=15, dog_low=1.0, dog_high=8.0,
                  min_size=8, len_um_hyphae=3.0, vox=0.145, max_panels=20):
    """Propose yeast|hyphae per candida object; SHOW overview + per-object panels + feature table
    so YOU confirm visually. hyphae = branched OR skeleton length >= len_um_hyphae."""
    raw = np.ascontiguousarray(get_view("ROI2","candida",t,correct=False).compute()).astype(np.float32)
    pre = preprocess_volume(raw, preprocess, tophat_radius=tophat_radius, dog_low=dog_low, dog_high=dog_high)
    thr = remove_small_objects(pre > threshold_otsu(pre), min_size)
    labels = cc_label(thr); skel = skeletonize(thr)
    feats = {}
    if skel.any():
        df = summarize(Skeleton(skel, spacing=(vox,vox,vox)), separator="_")
        for sid, g in df.groupby("skeleton_id"):
            L = float(g["branch_distance"].sum()); junc = bool((g["branch_type"] >= 1).any() and len(g) > 1)
            z,y,x = (int(g["image_coord_src_0"].iloc[0]), int(g["image_coord_src_1"].iloc[0]), int(g["image_coord_src_2"].iloc[0]))
            feats[int(labels[z,y,x])] = (round(L,2), junc, (L >= len_um_hyphae or junc))
    rows = []
    for prop in regionprops(labels):
        L, junc, ishy = feats.get(prop.label, (0.0, False, False))
        rows.append({"obj": prop.label, "vox": int(prop.area), "skel_len_um": L,
                     "branched": junc, "PROPOSED_class": "hyphae" if ishy else "yeast"})
    tbl = pd.DataFrame(rows).sort_values("vox", ascending=False)
    classv = np.zeros_like(labels, "uint8")
    for _, r in tbl.iterrows():
        classv[labels == r["obj"]] = 2 if r["PROPOSED_class"] == "hyphae" else 1
    # --- overview: raw | threshold | skeleton | PROPOSED class (confirm visually) ---
    cmap = mcolors.ListedColormap(["black", "#3b82f6", "#ef4444"])  # bg / yeast / hyphae
    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    ax[0].imshow(raw.max(0), cmap="gray"); ax[0].set_title(f"raw MIP t{t}")
    ax[1].imshow(thr.max(0), cmap="gray"); ax[1].set_title(f"threshold ({preprocess})")
    ax[2].imshow(thr.max(0), cmap="gray"); sy, sx = np.where(skel.max(0)); ax[2].scatter(sx, sy, s=0.5, c="red"); ax[2].set_title("skeleton")
    ax[3].imshow(classv.max(0), cmap=cmap, vmin=0, vmax=2); ax[3].set_title("PROPOSED class\n(blue=yeast, red=hyphae)")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
    # --- per-object panels: raw crop + mask contour + skeleton, so you judge each one ---
    props = sorted(regionprops(labels), key=lambda p: p.area, reverse=True)[:max_panels]
    if props:
        nc = 4; nr = int(np.ceil(len(props) / nc))
        fig, axes = plt.subplots(nr, nc, figsize=(4 * nc, 4 * nr)); axes = np.atleast_1d(axes).ravel()
        for a, prop in zip(axes, props):
            z0, y0, x0, z1, y1, x1 = prop.bbox
            a.imshow(raw[z0:z1, y0:y1, x0:x1].max(0), cmap="gray")
            a.contour((labels[z0:z1, y0:y1, x0:x1] == prop.label).max(0), colors="cyan", linewidths=0.6)
            ys, xs = np.where(skel[z0:z1, y0:y1, x0:x1].max(0)); a.scatter(xs, ys, s=1, c="red")
            L, junc, ishy = feats.get(prop.label, (0, False, False))
            a.set_title(f"obj {prop.label}: {L}um  br={junc}\n-> PROPOSED {'hyphae' if ishy else 'yeast'}", fontsize=9)
            a.axis("off")
        for a in axes[len(props):]: a.axis("off")
        plt.tight_layout(); plt.show()
    display(tbl)
    print("^ PROPOSED classes — confirm each object in the panels above before trusting them.")
    return labels, classv, tbl, skel

# EDIT + re-run. Confirm each panel. Try preprocess "tophat" vs "dog"; raise min_size to drop
# specks; len_um_hyphae = length above which an UNbranched object still counts as hyphal.
labels_c, class_c, tbl_c, skel_c = candida_split(t=90, preprocess="dog", min_size=5, len_um_hyphae=3.0)

## 3c — micro-sam candida: per-object diagnostics (PROPOSED)
Loads the saved `results/ROI2/candida_microsam.zarr` (from the A30 μSAM run) and shows the SAME per-object panels + feature table as the threshold prototype, so you can confirm each of the μSAM objects and compare. **PROPOSED — your eyes decide; μSAM captured ~11x more foreground than the threshold, but confirm it's real structure, not over-fragmentation/noise.**

In [ ]:
# Load the saved micro-sam masks (A30 run) + raw candida; show overview + per-object panels + table.
import zarr, matplotlib.colors as mcolors
from IPython.display import display

T_MS = 90
labels_ms = zarr.open_group("results/ROI2/candida_microsam.zarr", mode="r")[f"t{T_MS:03d}"][:]
raw_ms = np.ascontiguousarray(get_view("ROI2", "candida", T_MS, correct=False).compute()).astype(np.float32)
feats_ms, tbl_ms = object_features(labels_ms)                 # skeleton features per micro-sam object
n_hy = int((tbl_ms["PROPOSED_class"] == "hyphae").sum()); n_ye = len(tbl_ms) - n_hy
print(f"micro-sam candida t{T_MS:03d}: {len(tbl_ms)} objects -> PROPOSED yeast={n_ye} hyphae={n_hy} (confirm below)")

# overview: raw MIP | micro-sam labels MIP | PROPOSED class MIP
classv = np.zeros_like(labels_ms, "uint8")
for _, r in tbl_ms.iterrows(): classv[labels_ms == r["obj"]] = 2 if r["PROPOSED_class"] == "hyphae" else 1
cmap = mcolors.ListedColormap(["black", "#3b82f6", "#ef4444"])
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(raw_ms.max(0), cmap="gray"); ax[0].set_title(f"raw MIP t{T_MS}")
ax[1].imshow((labels_ms.max(0) > 0), cmap="gray"); ax[1].set_title(f"micro-sam labels ({len(tbl_ms)})")
ax[2].imshow(classv.max(0), cmap=cmap, vmin=0, vmax=2); ax[2].set_title("PROPOSED class\n(blue=yeast, red=hyphae)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

show_object_panels(raw_ms, labels_ms, feats_ms, title=f"micro-sam candida t{T_MS} — PROPOSED", max_panels=16)
display(tbl_ms)
print("^ PROPOSED — confirm each object in the panels; nothing here is validated.")

In [ ]:
# OPTIONAL full-3D view. In-notebook overlay above is usually enough; for true 3D use
# view_labels.py on a VNC GPU desktop instead:
#   pixi run python view_labels.py ROI2 candida 90
try:
    import napari
    v = napari.Viewer(ndisplay=3)
    v.add_image(np.ascontiguousarray(get_view("ROI2","candida",90,correct=False).compute()).astype("float32"),
                name="candida raw", scale=(0.145,0.145,0.145))
    v.add_labels(labels_cand, name="candida labels", scale=(0.145,0.145,0.145))
    napari.run()
except Exception as e:
    print("napari not shown (headless kernel has no display — use view_labels.py on VNC):", e)

## 5 — Next
Once params look right they're saved to `params/cpsam_*.json`; the batch arrays `segment_ROI2.sh` / `segment_ROI7.sh` read the same sidecars.